## AI Based Resume Screening System

### [Dataset](https://www.kaggle.com/datasets/ameyachopade/resume-screening-and-candidate-selection-dataset$0)

In [17]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
import warnings
from sklearn.preprocessing import OneHotEncoder
import joblib

In [18]:
df = pd.read_csv('/content/resume_screening_dataset.csv')
df.head()

,candidate_id,age,gender,degree,specialization,university_tier,cgpa,years_experience,internships_count,projects_count,...,deep_learning_skill,communication_skill,leadership_skill,github_projects,kaggle_activity_score,hackathons_participated,expected_salary_lpa,location,employment_gap_months,selected
0,CAND-000001,24,Female,B.Tech,Information Technology,Tier 3,8.75,6,1,2,...,7,6,4,3,6.1,2,16.5,Kolkata,0,No
1,CAND-000002,37,Male,B.E.,Mechanical,Tier 2,7.08,14,1,5,...,4,10,8,3,8.2,1,27.6,Delhi,3,No
2,CAND-000003,32,Female,Ph.D,Information Technology,Tier 1,7.08,10,0,6,...,7,6,4,5,6.0,2,22.1,Delhi,0,Yes
3,CAND-000004,28,Male,B.E.,Data Science,Tier 3,8.28,10,1,3,...,4,6,4,5,9.7,2,19.4,Pune,0,Yes
4,CAND-000005,25,Female,B.E.,Business Analytics,Tier 1,7.42,4,0,1,...,4,6,4,2,2.8,1,14.3,Mumbai,0,No


In [19]:
df.describe()

,age,cgpa,years_experience,internships_count,projects_count,certifications_count,python_skill,sql_skill,machine_learning_skill,deep_learning_skill,communication_skill,leadership_skill,github_projects,kaggle_activity_score,hackathons_participated,expected_salary_lpa,employment_gap_months
count,50000.000000,50000.000000,50000.000000,50000.000000,50000.000000,50000.000000,50000.00000,50000.000000,50000.000000,50000.000000,50000.000000,50000.000000,50000.000000,50000.000000,50000.000000,50000.000000,50000.000000
mean,29.086480,8.124736,7.560560,1.442200,3.896180,1.937500,7.37256,6.394680,6.396740,5.216380,6.405980,5.729120,3.200760,5.804350,1.978800,18.279176,1.458480
std,6.629926,0.806433,5.648703,1.223392,2.618629,1.632215,1.84004,1.733173,1.847952,1.696165,1.736946,1.884424,1.977325,1.461063,1.459408,8.779321,3.419595
min,18.000000,5.230000,0.000000,0.000000,0.000000,0.000000,1.00000,1.000000,1.000000,1.000000,1.000000,1.000000,0.000000,1.300000,0.000000,3.000000,0.000000
25%,23.000000,7.570000,2.000000,1.000000,2.000000,1.000000,6.00000,5.000000,5.000000,4.000000,5.000000,4.000000,2.000000,4.800000,1.000000,10.100000,0.000000
50%,29.000000,8.180000,8.000000,1.000000,4.000000,2.000000,7.00000,6.000000,6.000000,5.000000,6.000000,6.000000,3.000000,5.700000,2.000000,18.300000,0.000000
75%,35.000000,8.730000,13.000000,2.000000,6.000000,3.000000,9.00000,8.000000,8.000000,6.000000,8.000000,7.000000,4.000000,6.700000,3.000000,26.400000,1.000000
max,40.000000,9.940000,15.000000,8.000000,17.000000,12.000000,10.00000,10.000000,10.000000,10.000000,10.000000,10.000000,16.000000,10.000000,11.000000,35.000000,24.000000


In [20]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 24 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   candidate_id             50000 non-null  object 
 1   age                      50000 non-null  int64  
 2   gender                   50000 non-null  object 
 3   degree                   50000 non-null  object 
 4   specialization           50000 non-null  object 
 5   university_tier          50000 non-null  object 
 6   cgpa                     50000 non-null  float64
 7   years_experience         50000 non-null  int64  
 8   internships_count        50000 non-null  int64  
 9   projects_count           50000 non-null  int64  
 10  certifications_count     50000 non-null  int64  
 11  python_skill             50000 non-null  int64  
 12  sql_skill                50000 non-null  int64  
 13  machine_learning_skill   50000 non-null  int64  
 14  deep_learning_skill   

In [21]:
warnings.filterwarnings('ignore')

df = df.drop(['candidate_id','location'],axis=1)

X = df[['age','gender','degree','specialization','university_tier','cgpa','years_experience','internships_count','projects_count','certifications_count','python_skill','sql_skill','machine_learning_skill','deep_learning_skill','communication_skill','leadership_skill','github_projects', 'kaggle_activity_score', 'hackathons_participated', 'expected_salary_lpa','employment_gap_months']]
y = df['selected']

X['university_tier'] = X['university_tier'].map({'Tier 3': 3, 'Tier 2': 2,'Tier 1':1})
X['gender'] = X['gender'].map({'Female': 0, 'Male': 1})

categorical_columns = X.select_dtypes(include=['object']).columns
encoder = OneHotEncoder(sparse_output=False)

encoded_data = encoder.fit_transform(X[categorical_columns])

encoded_df = pd.DataFrame(
    encoded_data,
    columns=encoder.get_feature_names_out(categorical_columns)
)

X = pd.concat(
    [X.drop(columns=categorical_columns), encoded_df],
    axis=1
)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

rf_classifier = RandomForestClassifier(n_estimators=100, random_state=42)
rf_classifier.fit(X_train, y_train)

y_pred = rf_classifier.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
classification_rep = classification_report(y_test, y_pred)

print(f"Accuracy: {accuracy:.2f}")
print("\nClassification Report:\n", classification_rep)

sample = X_test.iloc[0:1]
prediction = rf_classifier.predict(sample)

sample_dict = sample.iloc[0].to_dict()
print(f"\nSample Candidate: {sample_dict}")
print(f"Predicted Selection: {'Selected' if prediction[0] == 1 else 'Not Selected'}")

Accuracy: 0.87

Classification Report:
               precision    recall  f1-score   support

          No       0.89      0.90      0.89      5990
         Yes       0.84      0.83      0.84      4010

    accuracy                           0.87     10000
   macro avg       0.87      0.87      0.87     10000
weighted avg       0.87      0.87      0.87     10000


Sample Candidate: {'age': 21.0, 'gender': 1.0, 'university_tier': 3.0, 'cgpa': 8.49, 'years_experience': 0.0, 'internships_count': 2.0, 'projects_count': 3.0, 'certifications_count': 1.0, 'python_skill': 5.0, 'sql_skill': 5.0, 'machine_learning_skill': 8.0, 'deep_learning_skill': 2.0, 'communication_skill': 5.0, 'leadership_skill': 6.0, 'github_projects': 2.0, 'kaggle_activity_score': 5.6, 'hackathons_participated': 2.0, 'expected_salary_lpa': 4.6, 'employment_gap_months': 0.0, 'degree_B.Com': 0.0, 'degree_B.E.': 0.0, 'degree_B.Sc': 0.0, 'degree_B.Tech': 0.0, 'degree_BBA': 1.0, 'degree_BCA': 0.0, 'degree_M.Sc': 0.0, 'degree_

In [22]:
joblib.dump(rf_classifier,'./rf.joblib')

['./rf.joblib']

In [24]:
model = joblib.load('/content/rf.joblib')
model.predict(X_test)

array(['No', 'No', 'Yes', ..., 'No', 'Yes', 'No'],
      shape=(10000,), dtype=object)